# GIT Setup


In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ADL/Project/Product-Search

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ADL/Project/Product-Search


In [4]:
#RUN AT START

!git config --global user.name "emdil99"
!git config --global user.email "emdilauro99@gmail.com"

from google.colab import userdata
token = userdata.get('github_key')
assert token is not None, "GitHub token not found"

repo_url = f"https://{token}@github.com/emdil99/Product-Search.git"
!git remote set-url origin $repo_url
!git pull origin main

print("Git remote updated securely using hidden token.")

From https://github.com/emdil99/Product-Search
 * branch            main       -> FETCH_HEAD
Already up to date.
Git remote updated securely using hidden token.


In [5]:
#RUN TO END CODE SESSION

!git add search_engine.ipynb

#UPDATE COMMIT COMMENT
!git commit -m "FAISS"


!git push origin main
print("Updated notebook on GitHub")

[main 8c2656c] FAISS
 1 file changed, 1 insertion(+), 1398 deletions(-)
 rewrite search_engine.ipynb (99%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 4.54 KiB | 178.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/emdil99/Product-Search.git
   212a3f4..8c2656c  main -> main
Updated notebook on GitHub


In [7]:
#!pip -q install sentence-transformers faiss-cpu
#!pip install -q pyarrow pandas
#!git clone https://github.com/amazon-science/esci-data.git
#!ls esci-data/shopping_queries_dataset

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer as ST
import faiss
import time
from tqdm import tqdm


# Data Structuring

In [9]:
#path = "esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet"
#df = pd.read_parquet(path, filters = [("product_locale", "==", "us")])
#df.columns
#df.head()
#df.to_parquet("eng_products.parquet", index=False)

eng_products = pd.read_parquet("eng_products.parquet")


In [ ]:
eng_products.head()



,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,None,Virtually silent at less than 0.3 sones\nPreci...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,None,Super quiet 80CFM energy efficient fan virtual...,Aero Pure,White,us
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",Aero Pure,White Finish,us
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...,None,Quiet operation at 1.5 Sones\nPrecision engine...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...,None,Ultra energy-efficient LED module (11-watt equ...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us


In [10]:
#from this data, I want to build the embeddings for the search function based off product title to start
#play around with bullet description in a later variation

products = eng_products[["product_id","product_title"]].drop_duplicates().copy()
products = products.rename(columns={"product_title":"product_text"})
title_by_id = dict(zip(products["product_id"], products["product_text"]))
products.head()

,product_id,product_text
0,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...
1,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...
2,B011RX6PNO,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...
3,B01MZIK0PI,Delta Electronics (Americas) Ltd. RAD80 Delta ...
4,B01N5Y6002,Delta Electronics (Americas) Ltd. GBR80HLED De...


# Embeddings

In [13]:
model = ST("sentence-transformers/all-mpnet-base-v2")

#sample_texts = products["product_text"].astype(str).head(5).tolist()
#sample_vecs = model.encode(sample_texts, normalize_embeddings=True)

#print("sample_vecs shape:", sample_vecs.shape)
#print("first vector, first 5 numbers:", sample_vecs[0][:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
texts = products["product_text"].astype(str).tolist()
batch_size = 256
all_vectors = []


for i in tqdm(range(0,len(texts), batch_size)):
  batch_texts = texts[i:i+batch_size]
  batch_vecs = model.encode(batch_texts, normalize_embeddings=True)
  all_vectors.append(batch_vecs)

product_emb = np.vstack(all_vectors).astype("float32")

np.save("product_emb.npy", product_emb)
products[["product_id"]].to_csv("product_id_map.csv", index=False)
print("done")

100%|██████████| 4750/4750 [13:46<00:00,  5.74it/s]


done


In [ ]:
!ls -lh product_emb.npy product_id_map.csv


-rw------- 1 root root 3.5G Jan  8 15:50 product_emb.npy
-rw------- 1 root root  13M Jan  8 15:50 product_id_map.csv


In [8]:

emb_path = "/content/drive/MyDrive/ADL/Project/Product-Search/product_emb.npy"
id_path  = "/content/drive/MyDrive/ADL/Project/Product-Search/product_id_map.csv"

product_emb = np.load(emb_path)
product_ids = pd.read_csv(id_path)

print("Embeddings:", product_emb.shape, product_emb.dtype)
print("IDs:", product_ids.shape)
print("Rows match?", product_emb.shape[0] == product_ids.shape[0])


Embeddings: (1215854, 768) float32
IDs: (1215854, 1)
Rows match? True


# FAISS

In [11]:
embedding_dim = product_emb.shape[1]
index_exact = faiss.IndexFlatIP(embedding_dim)
index_exact.add(product_emb)

def embed_query(query:str):
  return model.encode([query], normalize_embeddings = True).astype("float32")



def index_search(index, query:str, k: int = 5):
  query_vector = embed_query(query)
  t = time.time()
  D, I = index.search(query_vector, k)
  ms = (time.time() - t) * 1000

  results = []

  for rank, (index,score) in enumerate(zip(I[0].tolist(), D[0].tolist()), start=1):
    top_prod = product_ids.iloc[index]["product_id"]
    results.append({
            "rank": rank,
            "product_id": top_prod,
            "title": title_by_id.get(top_prod, ""),
            "score": float(score),
        })
  return results, ms




In [14]:
results, ms = index_search(
    index_exact,
    "wireless noise cancelling earbuds",
    k=5
)

ms, results


(594.6416854858398,
 [{'rank': 1,
   'product_id': 'B07N3RPPB7',
   'title': 'Audio-Technica ATH-ANC900BT QuietPoint Wireless Active Noise-Cancelling Headphones',
   'score': 0.8102340698242188},
  {'rank': 2,
   'product_id': 'B09B42NXQP',
   'title': 'Industry Leading Noise Canceling Truly Wireless Earbuds Headset/Headphones with Mic for iOS and Android Phones (Black)',
   'score': 0.8001851439476013},
  {'rank': 3,
   'product_id': 'B08G211JLR',
   'title': 'Skullcandy Indy ANC True Wireless Noise Cancelling In-Ear Earbud - True Black',
   'score': 0.7847440838813782},
  {'rank': 4,
   'product_id': 'B0832LTH73',
   'title': 'Bluetooth Sports Earbuds Wireless Earbuds Bluetooth 5.0 True Wireless Bluetooth Earbuds with Charging Case Noise Cancelling',
   'score': 0.767866849899292},
  {'rank': 5,
   'product_id': 'B00D429Y12',
   'title': 'Bose QuietComfort 20i Acoustic Noise Cancelling Headphones',
   'score': 0.7634381055831909}])